In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install gymnasium
!pip install stable-baselines3

In [ ]:
import os
import json
import random

import torch
import torch as th
import torch.nn as nn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym

from gymnasium import spaces

from stable_baselines3 import PPO

from stable_baselines3.common.env_util import make_vec_env

from stable_baselines3.common.vec_env import DummyVecEnv

from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

In [ ]:
STATE_TENSOR_PATH = (
    "/content/drive/MyDrive/"
    "PyroRL_Saudi_Project/"
    "datasets/saudi_eastern_province/"
    "grids/32x32/state_tensor.npy"
)

state_tensor = np.load(
    STATE_TENSOR_PATH
).astype(np.float32)

print("Tensor Shape:", state_tensor.shape)

In [ ]:
channel_names = [

    "Fire",
    "Fuel",
    "Wind X",
    "Wind Y",
    "Terrain",
    "Temperature",
    "Humidity"
]

fig, axes = plt.subplots(2,4, figsize=(15,8))

axes = axes.flatten()

for i in range(7):

    im = axes[i].imshow(state_tensor[i])

    axes[i].set_title(channel_names[i])

    plt.colorbar(im, ax=axes[i])

axes[7].axis('off')

plt.tight_layout()

plt.show()

In [ ]:
class SaudiWildfireEnv(gym.Env):

    def __init__(self):

        super(SaudiWildfireEnv, self).__init__()

        self.grid_size = 32

        self.max_steps = 200

        self.current_step = 0

        self.initial_tensor = np.load(
            STATE_TENSOR_PATH
        ).astype(np.float32)

        self.state = self.initial_tensor.copy()

        self.action_space = spaces.Discrete(5)

        self.observation_space = spaces.Box(

            low=0.0,
            high=1.0,

            shape=(7,32,32),

            dtype=np.float32
        )

        self.agent_pos = [16,16]

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.state = self.initial_tensor.copy()

        self.current_step = 0

        self.agent_pos = [16,16]

        return self.state, {}

    def step(self, action):

        self.current_step += 1

        x, y = self.agent_pos

        # ==========================
        # MOVEMENT
        # ==========================

        if action == 0:
            x = max(0, x-1)

        elif action == 1:
            x = min(self.grid_size-1, x+1)

        elif action == 2:
            y = max(0, y-1)

        elif action == 3:
            y = min(self.grid_size-1, y+1)

        self.agent_pos = [x,y]

        # ==========================
        # FIRE SUPPRESSION
        # ==========================

        for dx in [-1,0,1]:

            for dy in [-1,0,1]:

                nx = x + dx
                ny = y + dy

                if (
                    0 <= nx < self.grid_size and
                    0 <= ny < self.grid_size
                ):

                    self.state[0, nx, ny] *= 0.2

        # ==========================
        # FIRE SPREAD
        # ==========================

        new_fire = self.state[0].copy()

        for i in range(1, self.grid_size-1):

            for j in range(1, self.grid_size-1):

                if self.state[0, i, j] > 0.2:

                    neighbors = [

                        (i-1,j),
                        (i+1,j),
                        (i,j-1),
                        (i,j+1)
                    ]

                    for ni, nj in neighbors:

                        fuel = self.state[1, ni, nj]

                        wind_factor = (

                            self.state[2, ni, nj]
                            +
                            self.state[3, ni, nj]

                        ) / 2

                        terrain_factor = self.state[4, ni, nj]

                        spread_prob = (

                            0.01
                            +
                            0.15 * fuel
                            +
                            0.08 * wind_factor
                            +
                            0.08 * terrain_factor
                        )

                        if np.random.rand() < spread_prob:

                            new_fire[ni, nj] = min(
                                1.0,
                                new_fire[ni, nj] + 0.12
                            )

        self.state[0] = new_fire

        # ==========================
        # FIRE DECAY
        # ==========================

        self.state[0] *= 0.97

        self.state[0] = np.clip(
            self.state[0],
            0,
            1
        )

        self.state[0][
            self.state[0] < 0.02
        ] = 0

        # ==========================
        # FUEL DEPLETION
        # ==========================

        self.state[1] -= (
            self.state[0] * 0.003
        )

        self.state[1] = np.clip(
            self.state[1],
            0,
            1
        )

        # ==========================
        # REWARD
        # ==========================

        total_fire = np.sum(
            self.state[0]
        )

        suppression_bonus = 0

        if self.state[0, x, y] < 0.1:

            suppression_bonus = 2

        reward = (
            -total_fire
            +
            suppression_bonus
        )

        # ==========================
        # TERMINATION
        # ==========================

        done = False

        if self.current_step >= self.max_steps:

            done = True

        if total_fire < 0.1:

            done = True

        return (
            self.state,
            reward,
            done,
            False,
            {}
        )

In [ ]:
env = SaudiWildfireEnv()

obs, info = env.reset()

print("Observation Shape:", obs.shape)

print("Action Space:", env.action_space)

In [ ]:
class CustomCNN(BaseFeaturesExtractor):

    def __init__(
        self,
        observation_space,
        features_dim=256
    ):

        super(CustomCNN, self).__init__(
            observation_space,
            features_dim
        )

        n_input_channels = (
            observation_space.shape[0]
        )

        self.cnn = nn.Sequential(

            nn.Conv2d(
                n_input_channels,
                32,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            nn.Flatten()
        )

        with th.no_grad():

            sample = th.as_tensor(

                observation_space.sample()[None]

            ).float()

            n_flatten = self.cnn(
                sample
            ).shape[1]

        self.linear = nn.Sequential(

            nn.Linear(
                n_flatten,
                features_dim
            ),

            nn.ReLU()
        )

    def forward(self, observations):

        return self.linear(
            self.cnn(observations)
        )

In [ ]:
policy_kwargs = dict(

    features_extractor_class=CustomCNN,

    features_extractor_kwargs=dict(
        features_dim=256
    )
)

In [ ]:
SEEDS = [0,1,2,3,4]

TRAIN_TIMESTEPS = 100000

EVAL_EPISODES = 10

results = []

In [ ]:
MODEL_DIR = (
    "/content/drive/MyDrive/"
    "PyroRL_Saudi_Project/models"
)

RESULT_DIR = (
    "/content/drive/MyDrive/"
    "PyroRL_Saudi_Project/results"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

print("Directories Ready.")

In [ ]:
for seed in SEEDS:

    print("\n" + "="*30)

    print(f"===== Training Seed {seed} =====")

    print("="*30)

    # ===================================
    # RANDOM SEEDS
    # ===================================

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    # ===================================
    # VECTOR ENVIRONMENT
    # ===================================

    vec_env = make_vec_env(

        SaudiWildfireEnv,

        n_envs=1,

        vec_env_cls=DummyVecEnv,

        seed=seed
    )

    # ===================================
    # PPO MODEL
    # ===================================

    model = PPO(

        "CnnPolicy",

        vec_env,

        verbose=0,

        learning_rate=3e-4,

        n_steps=2048,

        batch_size=256,

        gamma=0.99,

        gae_lambda=0.95,

        clip_range=0.2,

        ent_coef=0.01,

        policy_kwargs=policy_kwargs,

        tensorboard_log=(
            f"./ppo_saudi_tensorboard_seed_{seed}/"
        ),

        seed=seed
    )

    # ===================================
    # TRAIN PPO
    # ===================================

    print("Training PPO...")

    model.learn(
        total_timesteps=TRAIN_TIMESTEPS
    )

    # ===================================
    # SAVE MODEL
    # ===================================

    model_path = (

        f"{MODEL_DIR}/"
        f"ppo_saudi_32x32_100k_seed_{seed}"
    )

    model.save(model_path)

    print(f"Model saved: {model_path}")

    # ===================================
    # EVALUATION
    # ===================================

    print("Evaluating model...")

    eval_rewards = []

    eval_burned_cells = []

    eval_fire_intensity = []

    for ep in range(EVAL_EPISODES):

        eval_env = SaudiWildfireEnv()

        obs, info = eval_env.reset(
            seed=seed + ep
        )

        done = False

        episode_rewards = []

        while not done:

            action, _ = model.predict(

                obs,

                deterministic=True
            )

            obs, reward, done, truncated, info = eval_env.step(action)

            episode_rewards.append(reward)

        # ===================================
        # FINAL STATES
        # ===================================

        final_fire_state = obs[0]

        # ===================================
        # METRICS
        # ===================================

        severe_burned_cells = int(

            np.sum(
                final_fire_state > 0.5
            )
        )

        fire_intensity = float(

            np.sum(final_fire_state)
        )

        total_episode_reward = float(

            np.sum(episode_rewards)
        )

        eval_rewards.append(
            total_episode_reward
        )

        eval_burned_cells.append(
            severe_burned_cells
        )

        eval_fire_intensity.append(
            fire_intensity
        )

    # ===================================
    # STORE RESULTS
    # ===================================

    results.append({

        "Seed": seed,

        "Mean Episode Reward":
        np.mean(eval_rewards),

        "Reward Std":
        np.std(eval_rewards),

        "Mean Burned Cells":
        np.mean(eval_burned_cells),

        "Burned Cells Std":
        np.std(eval_burned_cells),

        "Mean Fire Intensity":
        np.mean(eval_fire_intensity),

        "Fire Intensity Std":
        np.std(eval_fire_intensity)
    })

In [ ]:
print("\n--- MULTI-SEED EXPERIMENT COMPLETE ---\n")

In [ ]:
df_results = pd.DataFrame(results)

display(df_results)

In [ ]:
summary_stats = {

    "mean_episode_reward":
    float(
        df_results[
            "Mean Episode Reward"
        ].mean()
    ),

    "std_episode_reward":
    float(
        df_results[
            "Mean Episode Reward"
        ].std()
    ),

    "mean_burned_cells":
    float(
        df_results[
            "Mean Burned Cells"
        ].mean()
    ),

    "std_burned_cells":
    float(
        df_results[
            "Mean Burned Cells"
        ].std()
    ),

    "mean_fire_intensity":
    float(
        df_results[
            "Mean Fire Intensity"
        ].mean()
    ),

    "std_fire_intensity":
    float(
        df_results[
            "Mean Fire Intensity"
        ].std()
    )
}

print(
    json.dumps(
        summary_stats,
        indent=4
    )
)

In [ ]:
csv_path = (

    f"{RESULT_DIR}/"
    f"multi_seed_results_saudi.csv"
)

df_results.to_csv(

    csv_path,

    index=False
)

print(f"CSV saved:\n{csv_path}")

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    df_results["Seed"],

    df_results["Mean Episode Reward"]
)

plt.xlabel("Seed")

plt.ylabel("Mean Episode Reward")

plt.title(
    "Saudi PPO Reward Across Seeds"
)

plt.grid(axis='y')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    df_results["Seed"],

    df_results["Mean Burned Cells"]
)

plt.xlabel("Seed")

plt.ylabel("Mean Burned Cells")

plt.title(
    "Saudi Burned Cells Across Seeds"
)

plt.grid(axis='y')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    df_results["Seed"],

    df_results["Mean Fire Intensity"]
)

plt.xlabel("Seed")

plt.ylabel("Mean Fire Intensity")

plt.title(
    "Saudi Fire Intensity Across Seeds"
)

plt.grid(axis='y')

plt.show()

In [ ]:
print("""

==========================================================
SAUDI PPO MULTI-SEED TRAINING COMPLETE
==========================================================

Completed:

- PPO wildfire suppression
- 100k timesteps
- 5 random seeds
- multi-episode evaluation
- statistical aggregation
- publication-grade outputs

==========================================================

""")